# LLM Foundations — Snowflake Notebook Edition

Hands-on tour of tokens, context windows, embeddings, retrieval, decoding controls, sentiment, hallucination and prompting — adapted to run natively in **Snowflake Notebooks**.

Notes for this edition:
- Real embeddings come from **Snowflake Cortex**, so cosine-similarity values will differ slightly from the theory document (which used illustrative numbers); the rankings and lessons are the same.
- Cells marked *simulated* still simulate LLM responses deterministically — no LLM call is made.
- Run the setup cell first; every later cell depends on it.

# Execution Topic: Snowflake Notebook Environment Setup

This notebook runs inside **Snowflake Notebooks** (Workspaces / Container Runtime or Warehouse Runtime). No `!pip install` or internet access is required.

| Need | Colab version | Snowflake version (this notebook) |
|---|---|---|
| Session | none | `get_active_session()` (Snowpark) |
| Real embeddings | `sentence-transformers` download from Hugging Face | **Snowflake Cortex** `SNOWFLAKE.CORTEX.EMBED_TEXT_768` with `snowflake-arctic-embed-m-v1.5` (runs inside Snowflake, no egress) |
| Tokenizers (`cl100k_base`, `o200k_base`) | `tiktoken` from PyPI + file download | `tiktoken` if installed **and** its encoding files are reachable (EAI or a stage); otherwise an approximate built-in tokenizer is used and clearly labelled |
| Snowflake-model token counts | – | `SNOWFLAKE.CORTEX.COUNT_TOKENS` (extra column in Part 1) |

`numpy`, `pandas`, `matplotlib` and `scikit-learn` are pre-installed in Snowflake Notebooks.

**Optional – exact OpenAI tokenizers without internet:** download `cl100k_base.tiktoken` and `o200k_base.tiktoken` once (from `https://openaipublic.blob.core.windows.net/encodings/`), upload them to a stage, and set `TIKTOKEN_STAGE` below (e.g. `'@MY_DB.MY_SCHEMA.MY_STAGE'`).

**Optional – use the Hugging Face model instead of Cortex:** on Container Runtime with an External Access Integration for PyPI/Hugging Face, run `!pip install tiktoken sentence-transformers` and set `EMBEDDING_BACKEND = "sentence_transformers"`.

> Cortex AI functions must be available in your account/region (or cross-region inference enabled), and your role needs the `SNOWFLAKE.CORTEX_USER` database role.

In [ ]:
# ---------------------------------------------------------
# Step 1: Configuration (edit if needed)
# ---------------------------------------------------------
EMBEDDING_BACKEND = "cortex"                        # "cortex" (default, no internet) or "sentence_transformers"
CORTEX_EMBED_MODEL = "snowflake-arctic-embed-m-v1.5"  # 768-D Cortex embedding model
ST_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TIKTOKEN_STAGE = None                               # e.g. "@MY_DB.MY_SCHEMA.MY_STAGE" holding *.tiktoken files
CORTEX_COUNT_MODEL = "llama3.1-8b"                  # model used for SNOWFLAKE.CORTEX.COUNT_TOKENS

# ---------------------------------------------------------
# Step 2: Imports used throughout the notebook
# ---------------------------------------------------------
import math
import re
import json
import os
import hashlib
import shutil
import tempfile
from collections import Counter
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_colwidth", 110)

# ---------------------------------------------------------
# Step 3: Tokenizers - real tiktoken if possible, else an approximate fallback
# ---------------------------------------------------------
TIKTOKEN_URLS = {
    "cl100k_base": "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken",
    "o200k_base":  "https://openaipublic.blob.core.windows.net/encodings/o200k_base.tiktoken",
}

def _prime_tiktoken_cache_from_stage(stage):
    """Copy *.tiktoken files from a stage into tiktoken's cache (file name = sha1 of the URL)."""
    cache_dir = os.path.join(tempfile.gettempdir(), "tiktoken_cache")
    os.makedirs(cache_dir, exist_ok=True)
    os.environ["TIKTOKEN_CACHE_DIR"] = cache_dir
    download_dir = tempfile.mkdtemp()
    for name, url in TIKTOKEN_URLS.items():
        session.file.get(f"{stage.rstrip('/')}/{name}.tiktoken", download_dir)
        shutil.copy(os.path.join(download_dir, f"{name}.tiktoken"),
                    os.path.join(cache_dir, hashlib.sha1(url.encode()).hexdigest()))

class ApproxTokenizer:
    """Fallback used ONLY when tiktoken encodings cannot be loaded.
    Splits text into space-prefixed words / digit groups / punctuation, then cuts long
    words into sub-word pieces. Same interface as tiktoken; counts are APPROXIMATE."""
    PATTERN = re.compile(r" ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+|\s+", re.UNICODE)

    def __init__(self, name, piece_len):
        self.name, self.piece_len = f"approx_{name}", piece_len
        self._vocab, self._inv = {}, {}

    def _id(self, piece):
        if piece not in self._vocab:
            idx = len(self._vocab) + 1000
            self._vocab[piece], self._inv[idx] = idx, piece
        return self._vocab[piece]

    def _pieces(self, text):
        for m in self.PATTERN.findall(text):
            is_ascii_word = m.strip().isascii() and m.strip().isalpha()
            size = self.piece_len if is_ascii_word else 2       # non-Latin scripts split finer
            if len(m.strip()) <= size:
                yield m
            else:
                for i in range(0, len(m), size):
                    yield m[i:i + size]

    def encode(self, text):
        return [self._id(p) for p in self._pieces(text)]

    def decode(self, ids):
        return "".join(self._inv[i] for i in ids)

    def decode_single_token_bytes(self, token_id):
        return self._inv[token_id].encode("utf-8")

TOKENIZER_IS_EXACT = False
try:
    import tiktoken
    if TIKTOKEN_STAGE:
        _prime_tiktoken_cache_from_stage(TIKTOKEN_STAGE)
    enc_cl100k = tiktoken.get_encoding("cl100k_base")   # e.g. GPT-4 / GPT-3.5 family
    enc_o200k = tiktoken.get_encoding("o200k_base")     # e.g. GPT-4o family
    TOKENIZER_IS_EXACT = True
except Exception as err:
    print(f"⚠️ tiktoken encodings not available ({type(err).__name__}); using the approximate fallback tokenizer.")
    enc_cl100k = ApproxTokenizer("cl100k_base", piece_len=6)
    enc_o200k = ApproxTokenizer("o200k_base", piece_len=8)

def cortex_count_tokens(text, model=CORTEX_COUNT_MODEL):
    """Exact token count for a Snowflake Cortex model (runs inside Snowflake)."""
    try:
        return session.sql("SELECT SNOWFLAKE.CORTEX.COUNT_TOKENS(?, ?) AS N", params=[model, text]).collect()[0]["N"]
    except Exception:
        return None

# ---------------------------------------------------------
# Step 4: Embedding model - Snowflake Cortex (default) or sentence-transformers
# Exposes .encode(...) like SentenceTransformer so later cells run unchanged.
# ---------------------------------------------------------
class CortexEmbeddingModel:
    def __init__(self, model=CORTEX_EMBED_MODEL, dim=768):
        self.model, self.dim, self._cache = model, dim, {}

    def _fetch(self, texts):
        df = session.create_dataframe([[i, t] for i, t in enumerate(texts)], schema=["IDX", "TXT"])
        expr = F.sql_expr(f"SNOWFLAKE.CORTEX.EMBED_TEXT_768('{self.model}', TXT)::ARRAY")
        rows = df.select("IDX", expr.alias("E")).order_by("IDX").collect()
        return [np.array(json.loads(r["E"]) if isinstance(r["E"], str) else r["E"], dtype=float) for r in rows]

    def encode(self, texts, convert_to_numpy=True, normalize_embeddings=False, **kwargs):
        single = isinstance(texts, str)
        texts = [texts] if single else list(texts)
        missing = [t for t in dict.fromkeys(texts) if t not in self._cache]   # cache avoids repeat Cortex calls
        if missing:
            self._cache.update(zip(missing, self._fetch(missing)))
        vectors = np.vstack([self._cache[t] for t in texts])
        if normalize_embeddings:
            vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
        return vectors[0] if single else vectors

    def get_embedding_dimension(self):
        return self.dim

    get_sentence_embedding_dimension = get_embedding_dimension

if EMBEDDING_BACKEND == "sentence_transformers":
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(ST_MODEL_NAME)
    EMBEDDING_LABEL = ST_MODEL_NAME
else:
    embedding_model = CortexEmbeddingModel()
    EMBEDDING_LABEL = f"Cortex {CORTEX_EMBED_MODEL}"

# ---------------------------------------------------------
# Step 5: Confirm the environment
# ---------------------------------------------------------
def _v(lib):
    try:
        return version(lib)
    except PackageNotFoundError:
        return "not installed"

print("Environment ready ✅")
print(f"  Snowflake account/role : {session.get_current_account()} / {session.get_current_role()}")
for lib in ["numpy", "pandas", "matplotlib", "scikit-learn", "snowflake-snowpark-python", "tiktoken"]:
    print(f"  {lib:<26} {_v(lib)}")
print(f"  Tokenizer              : {'tiktoken (exact)' if TOKENIZER_IS_EXACT else 'approximate fallback'}")
test_vec = embedding_model.encode(["hello snowflake"])
print(f"  Embedding model        : {EMBEDDING_LABEL} -> {test_vec.shape[1]}-D (Cortex call OK)")

## Part 1 — Tokens and Tokenization

# Execution Topic: Token Counting

A token is the smallest unit of text an LLM reads and writes: a word, part of a word, a space-prefixed word, punctuation or a digit group. Every limit and every bill is measured in tokens, and **tokenizers are model-specific**.

This execution demonstrates counting tokens with two real tokenizers (`cl100k_base` and `o200k_base`, both OpenAI encodings) and connects it to the English rule of thumb **1 token ≈ 4 characters ≈ 0.75 words** (a rough guide only). The output will show that token counts differ from characters/words, differ between tokenizers, and grow for Hindi/Kannada text.

In [ ]:
# ---------------------------------------------------------
# Step 1: Load two REAL tokenizers.
# Both are OpenAI encodings. Other model families (Claude, Gemini,
# Llama, ...) use their own tokenizers, so counts will differ there.
# ---------------------------------------------------------
# enc_cl100k / enc_o200k were created in the setup cell (tiktoken or approximate fallback).
print("Tokenizer mode:", "tiktoken (exact)" if TOKENIZER_IS_EXACT else "APPROXIMATE fallback - counts are indicative only")

# ---------------------------------------------------------
# Step 2: Same customer question in English, Hindi and Kannada
# ---------------------------------------------------------
samples = [
    ("English", "Where is my order?"),
    ("Hindi",   "मेरा ऑर्डर कहाँ है?"),
    ("Kannada", "ನನ್ನ ಆರ್ಡರ್ ಎಲ್ಲಿದೆ?"),
    ("English", "Refund not received, unbelievable!"),
    ("English", "Paid ₹500 but only got ₹200 back, this is ridiculous!"),
]

# ---------------------------------------------------------
# Step 3: Compare real token counts with the rough rule of thumb
# ---------------------------------------------------------
rows = []
for language, text in samples:
    rows.append({
        "Language": language,
        "Text": text,
        "Characters": len(text),
        "Words": len(text.split()),
        "Tokens cl100k_base": len(enc_cl100k.encode(text)),
        "Tokens o200k_base": len(enc_o200k.encode(text)),
        f"Tokens Cortex {CORTEX_COUNT_MODEL}": cortex_count_tokens(text),
        "Rule of thumb chars/4": round(len(text) / 4, 1),
        "Rule of thumb words/0.75": round(len(text.split()) / 0.75, 1),
    })
token_table = pd.DataFrame(rows)
display(token_table)

# ---------------------------------------------------------
# Step 4: Token multiplier of Indic scripts vs English (same question)
# ---------------------------------------------------------
english_tokens = token_table.loc[0, "Tokens cl100k_base"]
print("Same question, relative token count on cl100k_base:")
for i in [1, 2]:
    lang = token_table.loc[i, "Language"]
    ratio = token_table.loc[i, "Tokens cl100k_base"] / english_tokens
    print(f"  {lang:<8}: {ratio:.1f}x the English tokens  -> {ratio:.1f}x the cost for this text")
print("\nNote: the rule of thumb is for English only - it clearly under-estimates Indic scripts.")

# ---------------------------------------------------------
# Step 5: Visual comparison of the two tokenizers
# ---------------------------------------------------------
labels = [f"{r['Language']} #{i+1}" for i, r in token_table.iterrows()]
x = np.arange(len(labels))
plt.figure(figsize=(9, 3.5))
plt.bar(x - 0.2, token_table["Tokens cl100k_base"], width=0.4, label="cl100k_base")
plt.bar(x + 0.2, token_table["Tokens o200k_base"], width=0.4, label="o200k_base")
plt.xticks(x, labels)
plt.ylabel("Tokens")
plt.title("Same text, different tokenizers -> different token counts")
plt.legend()
plt.tight_layout()
plt.show()

# Execution Topic: Token Cost Calculation

LLM APIs bill per million tokens, and output tokens usually cost 3–5× input tokens.

$$\text{Cost} = \frac{T_{in} \times Price_{in} + T_{out} \times Price_{out}}{1{,}000{,}000}$$

where $T_{in}$, $T_{out}$ are input/output tokens and prices are per million tokens.

This execution demonstrates the DOCX support-bot example (50,000 tickets/day, 800 input + 150 output tokens per call, **illustrative** prices $3/M input and $15/M output) and connects it to prompt design. The output will show daily/monthly cost and the saving from cutting the system prompt by 300 tokens.

In [ ]:
# ---------------------------------------------------------
# Step 1: Inputs from the DOCX worked example
# Prices are ILLUSTRATIVE (per million tokens), as in the source document.
# ---------------------------------------------------------
tickets_per_day = 50_000
input_tokens_per_call = 800
output_tokens_per_call = 150
price_input_per_million = 3.00     # illustrative $ / 1M input tokens
price_output_per_million = 15.00   # illustrative $ / 1M output tokens
days_per_month = 30

def daily_token_cost(calls, t_in, t_out, p_in=price_input_per_million, p_out=price_output_per_million):
    """Cost = (T_in x Price_in + T_out x Price_out) / 1,000,000, summed over all calls."""
    daily_in_tokens = calls * t_in
    daily_out_tokens = calls * t_out
    in_cost = daily_in_tokens * p_in / 1_000_000
    out_cost = daily_out_tokens * p_out / 1_000_000
    return daily_in_tokens, daily_out_tokens, in_cost, out_cost

# ---------------------------------------------------------
# Step 2: Baseline cost
# ---------------------------------------------------------
d_in, d_out, c_in, c_out = daily_token_cost(tickets_per_day, input_tokens_per_call, output_tokens_per_call)
total_daily = c_in + c_out
print("BASELINE (illustrative prices)")
print(f"  Daily input tokens : {d_in:>13,}  -> input cost  ${c_in:,.2f}/day")
print(f"  Daily output tokens: {d_out:>13,}  -> output cost ${c_out:,.2f}/day")
print(f"  Total daily cost   : ${total_daily:,.2f}")
print(f"  Approx monthly cost: ${total_daily * days_per_month:,.2f}  (≈ $7,000 / month)")

# ---------------------------------------------------------
# Step 3: Cut the system prompt by 300 tokens (800 -> 500 input tokens)
# ---------------------------------------------------------
tokens_removed = 300
d_in2, _, c_in2, _ = daily_token_cost(tickets_per_day, input_tokens_per_call - tokens_removed, output_tokens_per_call)
print(f"\nAFTER removing {tokens_removed} tokens from the system prompt")
print(f"  Tokens saved per day : {d_in - d_in2:,}")
print(f"  Money saved per day  : ${c_in - c_in2:,.2f}")
print(f"  Money saved per month: ${(c_in - c_in2) * days_per_month:,.2f}  - just by editing text")

# Execution Topic: Token Latency

Generation is sequential: after the first token arrives, each output token adds more time.

$$\text{Latency} \approx \text{Time-to-first-token} + T_{out} \times \text{(time per output token)}$$

This execution demonstrates the latency formula with **illustrative** timing values (not a benchmark of any model) and connects it to the DOCX e-commerce case of capping `max_tokens`. The output will show latency growing linearly with output tokens.

In [ ]:
# ---------------------------------------------------------
# Step 1: Illustrative timing values (NOT measured from a real model)
# ---------------------------------------------------------
time_to_first_token_s = 0.5      # seconds (illustrative)
time_per_output_token_s = 0.02   # seconds per token (illustrative)

# ---------------------------------------------------------
# Step 2: Latency for different output lengths (e.g. different max_tokens caps)
# ---------------------------------------------------------
output_lengths = [50, 150, 500, 1000]
latency_table = pd.DataFrame({
    "Output tokens": output_lengths,
    "Latency (s)": [time_to_first_token_s + t * time_per_output_token_s for t in output_lengths],
})
display(latency_table)

print("Observation: latency is dominated by OUTPUT tokens.")
print("Capping max_tokens from 1000 to 150 cuts latency from "
      f"{latency_table.iloc[3, 1]:.1f}s to {latency_table.iloc[1, 1]:.1f}s in this illustration.")

# Execution Topic: Tokenization (Text → Token IDs → Text)

Tokenization is the deterministic process that converts raw text into token IDs (and back) before the model sees anything. The model never sees letters, only integers.

This execution demonstrates encoding and decoding "Refund not received, unbelievable!" with a real tokenizer and connects it to sub-word tokenization. The output will show the token IDs, the text piece behind each ID (`·` marks a leading space), and a perfect round trip.

In [ ]:
# ---------------------------------------------------------
# Step 1: Encode text into token IDs (cl100k_base tokenizer)
# ---------------------------------------------------------
text = "Refund not received, unbelievable!"
token_ids = enc_cl100k.encode(text)

# ---------------------------------------------------------
# Step 2: Look up the text piece behind every ID.
# '·' is used to make the leading space visible.
# ---------------------------------------------------------
def token_pieces(encoder, text):
    ids = encoder.encode(text)
    return [encoder.decode_single_token_bytes(i).decode("utf-8", errors="replace").replace(" ", "·") for i in ids]

pieces = token_pieces(enc_cl100k, text)

print("Original text:\n ", text)
print("\nToken IDs:\n ", token_ids)
print("\nDecoded tokens:\n ", pieces)
print("\nNumber of tokens:", len(token_ids), "  |  Number of words:", len(text.split()))

# ---------------------------------------------------------
# Step 3: Decode back - tokenization is lossless and deterministic
# ---------------------------------------------------------
round_trip = enc_cl100k.decode(token_ids)
print("\nDecoded back to text:", round_trip, "| identical:", round_trip == text)

# ---------------------------------------------------------
# Step 4: Same text, different tokenizer -> different IDs/pieces
# ---------------------------------------------------------
print("\no200k_base pieces:", token_pieces(enc_o200k, text))
print("Note: the DOCX figure's IDs are illustrative; real IDs depend on the tokenizer.")

# Execution Topic: Numbers, Spaces and Rare Words

Numbers and code split oddly, `" refund"` and `"refund"` are different tokens, and rare words or typos are built from reusable pieces.

This execution demonstrates tokenizer edge cases listed in the DOCX and connects them to engineering practice (count with the target model's tokenizer). The output will show dates and long numbers splitting into several tokens, different IDs for space-prefixed words, and misspellings still sharing pieces with the correct word.

In [ ]:
# ---------------------------------------------------------
# Step 1: Edge-case strings from the DOCX
# ---------------------------------------------------------
edge_cases = [
    "2024-09-18",                                  # date
    "1234567890",                                  # long number
    "refund",                                      # no leading space
    " refund",                                     # leading space
    "iphone", "iphnoe",                            # typo vs correct word
    "samsung", "samsng",
    "3f2b9c1e-7a4d-4e8b-9f10-2c3d4e5f6a7b",        # UUID (token-heavy logs)
]

# ---------------------------------------------------------
# Step 2: Tokenize each case and show IDs + pieces
# ---------------------------------------------------------
rows = []
for s in edge_cases:
    ids = enc_cl100k.encode(s)
    rows.append({"Text": repr(s), "Token count": len(ids), "Token IDs": ids,
                 "Pieces": token_pieces(enc_cl100k, s)})
display(pd.DataFrame(rows))

# ---------------------------------------------------------
# Step 3: Highlight the space effect
# ---------------------------------------------------------
print("'refund'  ->", enc_cl100k.encode("refund"))
print("' refund' ->", enc_cl100k.encode(" refund"), " <- different token(s): spaces matter!")

# Execution Topic: BPE Demonstration

Byte-Pair Encoding builds the vocabulary by repeatedly merging the most frequent adjacent pair:

$$(a^*, b^*) = \arg\max_{(a,b)} \text{count}(a, b) \;\rightarrow\; \text{add new token } ab$$

This execution demonstrates a tiny educational BPE (not a production tokenizer) on the DOCX corpus "low", "lower", "lowest" and connects it to how frequent strings become single tokens. The output will show pair counts at each step, the chosen merge, and the final segmentation `[low, e, r]` and `[low, e, s, t]`.

In [ ]:
# ---------------------------------------------------------
# Step 1: Start with every word split into characters
# ---------------------------------------------------------
corpus = ["low", "lower", "lowest"]
words = [list(w) for w in corpus]
vocabulary = sorted({ch for w in words for ch in w})
print("Start symbols:", words)
print("Initial vocabulary:", vocabulary)

def count_adjacent_pairs(words):
    """Count every adjacent symbol pair across the corpus."""
    counts = Counter()
    for w in words:
        for a, b in zip(w, w[1:]):
            counts[(a, b)] += 1
    return counts

def merge_pair(words, pair):
    """Replace every occurrence of the pair (a, b) with the merged symbol 'ab'."""
    merged_words = []
    for w in words:
        new_w, i = [], 0
        while i < len(w):
            if i < len(w) - 1 and (w[i], w[i + 1]) == pair:
                new_w.append(w[i] + w[i + 1]); i += 2
            else:
                new_w.append(w[i]); i += 1
        merged_words.append(new_w)
    return merged_words

# ---------------------------------------------------------
# Step 2: Repeat "count pairs -> merge most frequent" (2 merges as in the DOCX)
# Ties are broken by first occurrence, so (l, o) is chosen before (o, w).
# ---------------------------------------------------------
NUM_MERGES = 2   # try 4 or 5 to see "lowe", "lower" appear
for step in range(1, NUM_MERGES + 1):
    pair_counts = count_adjacent_pairs(words)
    best_pair = max(pair_counts, key=pair_counts.get)
    print(f"\n--- Merge {step} ---")
    print("Pair counts:", {f"{a}+{b}": c for (a, b), c in pair_counts.most_common()})
    print(f"Most frequent pair: {best_pair} (count {pair_counts[best_pair]}) -> new token '{''.join(best_pair)}'")
    words = merge_pair(words, best_pair)
    vocabulary.append("".join(best_pair))
    print("Words now:", words)

# ---------------------------------------------------------
# Step 3: Final result
# ---------------------------------------------------------
print("\nFinal vocabulary:", vocabulary)
for original, segmented in zip(corpus, words):
    print(f"  {original:<7} -> {segmented}")

## Part 2 — Context and Context Window

# Execution Topic: Context Changes the Answer

Context is all the information in a single request: system instructions, history, retrieved documents, tool results and the question. The model is stateless, so anything it should "know" about this customer must be in the context.

This execution demonstrates the DOCX question "Can I return this?" asked with and without order/policy context, and connects it to the idea that the same model gives different answers when only the context changes. The output will show a generic guess versus a correct, specific answer.

> **Note:** This is an educational simulation of an LLM response. No external LLM API is being called.

In [ ]:
# ---------------------------------------------------------
# Step 1: A mock "LLM" (deterministic rules) - SIMULATION ONLY
# Without context it falls back to a generic "parametric" guess;
# with context it uses the facts it was given.
# ---------------------------------------------------------
def mock_llm_answer(question, context=None):
    if context is None:
        return "Most stores allow returns within 30 days, so you can probably return it."
    window = context["return_window_days"][context["category"]]
    days = context["days_since_delivery"]
    if days <= window:
        return f"Yes - it was delivered {days} days ago, within the {window}-day window for {context['category']}."
    return (f"This item is outside the {window}-day window for {context['category']} "
            f"(delivered {days} days ago), but you can raise a warranty claim.")

def build_prompt(question, context=None):
    """Show what would actually be sent to the model."""
    if context is None:
        return f"User: {question}"
    return (f"System: Answer using the order record and policy below.\n"
            f"Order record: delivered {context['days_since_delivery']} days ago, category = {context['category']}\n"
            f"Policy: {context['category']} return window = {context['return_window_days'][context['category']]} days\n"
            f"User: {question}")

question = "Can I return this?"
order_context = {
    "days_since_delivery": 12,
    "category": "electronics",
    "return_window_days": {"electronics": 10},
}

# ---------------------------------------------------------
# Step 2: Question alone vs Question + relevant context
# ---------------------------------------------------------
for label, ctx in [("A) QUESTION ALONE", None), ("B) QUESTION + CONTEXT", order_context)]:
    print("=" * 70)
    print(label)
    print("-" * 70)
    print(build_prompt(question, ctx))
    print("\n[Simulated answer]", mock_llm_answer(question, ctx))
print("=" * 70)
print("Same 'model', same question - only the context changed.")

# Execution Topic: Context as Conditional Probability (Toy Example)

An LLM is a conditional probability machine: every output token is conditioned on the context plus everything generated so far.

$$P(y_1, \dots, y_m \mid \text{context}) = \prod_{t=1}^{m} P(y_t \mid \text{context}, y_1 \dots y_{t-1})$$

This execution demonstrates the product of next-token probabilities using an **educational toy example** with hand-written probabilities (not from a real LLM). The output will show how the same answer becomes far more likely once the policy is in the context.

In [ ]:
# ---------------------------------------------------------
# Step 1: Educational toy example - hand-written probabilities, NOT a real LLM
# Answer tokens for "...outside the 10-day window..."
# ---------------------------------------------------------
answer_tokens = ["outside", "the", "10", "-day", "window"]
p_without_policy = np.array([0.05, 0.90, 0.02, 0.40, 0.30])   # model must guess the policy
p_with_policy    = np.array([0.80, 0.95, 0.90, 0.95, 0.85])   # policy is in the context

# ---------------------------------------------------------
# Step 2: Chain rule - multiply the conditional next-token probabilities
# ---------------------------------------------------------
toy_table = pd.DataFrame({
    "Token y_t": answer_tokens,
    "P(y_t | no policy, y_<t)": p_without_policy,
    "Cumulative (no policy)": np.cumprod(p_without_policy),
    "P(y_t | with policy, y_<t)": p_with_policy,
    "Cumulative (with policy)": np.cumprod(p_with_policy),
})
display(toy_table.round(5))

p_seq_without = np.prod(p_without_policy)
p_seq_with = np.prod(p_with_policy)
print(f"P(answer | context without policy) = {p_seq_without:.6f}")
print(f"P(answer | context with policy)    = {p_seq_with:.6f}")
print(f"The correct answer is ~{p_seq_with / p_seq_without:,.0f}x more likely once the policy is in context.")
print(f"(Log form used in practice: sum of log-probs = {np.log(p_with_policy).sum():.3f})")

# Execution Topic: Context Window Budget

The context window is shared by the input and the output, so the hard constraint is:

$$T_{input} + T_{output} \le N_{max}$$

This execution demonstrates budgeting a 128K window (split taken from the DOCX figure) and connects it to the DOCX 200-page contract example (≈100,000 words ≈ 133,000 tokens). The output will show the free space in the budget and why the contract does not fit once output is reserved.

In [ ]:
# ---------------------------------------------------------
# Step 1: A 128K window budget (values from the DOCX figure)
# ---------------------------------------------------------
N_MAX = 128_000
input_budget = {"System prompt": 2_000, "Chat history": 10_000, "RAG chunks": 40_000, "User question": 1_000}
output_reserve = 8_000

def fits_in_window(input_tokens, output_tokens, n_max=N_MAX):
    return input_tokens + output_tokens <= n_max

total_input = sum(input_budget.values())
print(f"Input tokens: {total_input:,} | Output reserved: {output_reserve:,} | "
      f"Free: {N_MAX - total_input - output_reserve:,}")
print("Fits?", fits_in_window(total_input, output_reserve))

# ---------------------------------------------------------
# Step 2: The 200-page contract (words -> tokens using 0.75 words/token)
# ---------------------------------------------------------
contract_words = 100_000
contract_tokens_estimate = contract_words / 0.75
contract_tokens = 133_000   # DOCX rounded value
print(f"\nContract: {contract_words:,} words / 0.75 ≈ {contract_tokens_estimate:,.0f} tokens (DOCX uses ≈ {contract_tokens:,})")
needed = contract_tokens + output_reserve
print(f"Contract + output reserve = {needed:,} tokens vs window {N_MAX:,}")
print("Fits?", fits_in_window(contract_tokens, output_reserve), f"-> over by {needed - N_MAX:,} tokens")
print("Options: larger-window model | chunk + retrieve (Top-N) | summarise section by section")

# ---------------------------------------------------------
# Step 3: Visualise the window as one bar
# ---------------------------------------------------------
parts = list(input_budget.items()) + [("Output (reserved)", output_reserve),
                                      ("Free", N_MAX - total_input - output_reserve)]
left = 0
plt.figure(figsize=(10, 1.8))
for name, size in parts:
    plt.barh(0, size, left=left, edgecolor="white")
    plt.text(left + size / 2, 0, f"{name}\n{size // 1000}K", ha="center", va="center", fontsize=8)
    left += size
plt.xlim(0, N_MAX); plt.yticks([]); plt.xlabel("Tokens")
plt.title("Context window = 128K tokens (input + output must fit together)")
plt.tight_layout(); plt.show()

# Execution Topic: Chunking with Overlap

When a document is too long, split it into overlapping chunks and retrieve only the relevant ones.

$$\text{Chunks needed} = \left\lceil \frac{L - \text{overlap}}{\text{chunk\_size} - \text{overlap}} \right\rceil$$

where $L$ is the document length in tokens. This execution demonstrates the formula with the DOCX values (L = 133,000, chunk size = 1,000, overlap = 100) and verifies it by actually generating the chunk boundaries. The output will show 148 chunks from both methods, plus a small word-level example that makes the overlap visible.

In [ ]:
# ---------------------------------------------------------
# Step 1: Formula from the DOCX
# ---------------------------------------------------------
L, chunk_size, overlap = 133_000, 1_000, 100
chunks_by_formula = math.ceil((L - overlap) / (chunk_size - overlap))
print(f"ceil(({L:,} - {overlap}) / ({chunk_size:,} - {overlap})) = ceil({(L - overlap) / (chunk_size - overlap):.2f}) = {chunks_by_formula}")

# ---------------------------------------------------------
# Step 2: Verify by generating real chunk boundaries (sliding window)
# ---------------------------------------------------------
def make_chunk_ranges(length, size, overlap):
    ranges, start = [], 0
    while True:
        end = min(start + size, length)
        ranges.append((start, end))
        if end >= length:
            return ranges
        start += size - overlap        # step forward, keeping `overlap` tokens

chunk_ranges = make_chunk_ranges(L, chunk_size, overlap)
print(f"Chunks by simulation: {len(chunk_ranges)}  | matches formula: {len(chunk_ranges) == chunks_by_formula}")
print("First chunks:", chunk_ranges[:3], " ... Last chunk:", chunk_ranges[-1])

# ---------------------------------------------------------
# Step 3: Small word-level example so the overlap is visible
# ---------------------------------------------------------
sentence = "refunds are processed within seven business days after the returned item passes the quality check".split()
for s, e in make_chunk_ranges(len(sentence), size=6, overlap=2):
    print(f"  words {s:>2}-{e - 1:<2}: {' '.join(sentence[s:e])}")
print("The last 2 words of each chunk repeat at the start of the next -> no idea is cut in half.")

# Execution Topic: Attention Scaling (n²)

Self-attention compares every token with every other token, so work grows roughly quadratically.

$$\text{Attention cost} \propto n^2 \quad\Rightarrow\quad \left(\frac{128K}{8K}\right)^2 = 16^2 = 256\times$$

This execution demonstrates the relative pairwise work for 8K–128K tokens and connects it to why bigger context is not always better (cost, latency, "lost in the middle"). It is an **attention-complexity intuition, not a benchmark** of any specific model. The output will show 16× more tokens producing 256× more pairwise work.

In [ ]:
# ---------------------------------------------------------
# Step 1: Relative pairwise comparisons n x n, baseline 8K
# ---------------------------------------------------------
context_sizes = [8_000, 16_000, 32_000, 64_000, 128_000]
base = context_sizes[0]
attention_table = pd.DataFrame({
    "Context tokens (n)": context_sizes,
    "Pairwise comparisons (n²)": [n * n for n in context_sizes],
    "Tokens vs 8K": [n / base for n in context_sizes],
    "Attention work vs 8K": [(n / base) ** 2 for n in context_sizes],
})
display(attention_table.style.format({"Context tokens (n)": "{:,}", "Pairwise comparisons (n²)": "{:,.0f}",
                                      "Tokens vs 8K": "{:.0f}x", "Attention work vs 8K": "{:.0f}x"}))
print(f"(128K / 8K)^2 = {(128_000 / 8_000) ** 2:.0f}x more pairwise work")

# ---------------------------------------------------------
# Step 2: Linear growth (tokens) vs quadratic growth (attention)
# ---------------------------------------------------------
plt.figure(figsize=(7, 3.5))
plt.plot([n // 1000 for n in context_sizes], attention_table["Tokens vs 8K"], "o-", label="Tokens (linear)")
plt.plot([n // 1000 for n in context_sizes], attention_table["Attention work vs 8K"], "o-", label="Attention work (n²)")
plt.xlabel("Context size (K tokens)"); plt.ylabel("Relative to 8K")
plt.title("Attention-complexity intuition (not a model benchmark)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Part 3 — Embeddings and Retrieval

# Execution Topic: Embedding (Text → Vector)

An embedding maps text to a fixed-length vector, $E: \text{text} \rightarrow \mathbb{R}^d$, where similar meanings point in similar directions.

This execution demonstrates creating real embeddings with **Snowflake Cortex** (`SNOWFLAKE.CORTEX.EMBED_TEXT_768`, model `snowflake-arctic-embed-m-v1.5`, computed inside Snowflake) and connects it to "search by meaning, not by keywords". The output will show the number of embeddings, the dimension $d$ (768 for this model; `all-MiniLM-L6-v2` uses 384, other models 1024 or 1536), a sample vector, and that "I want my money returned" is closest to refund-type phrases despite sharing no keywords with them.

> The same call in SQL: `SELECT SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', 'I want my money returned');`

In [ ]:
# ---------------------------------------------------------
# Step 1: The embedding model was created in the setup cell.
# Default: Snowflake Cortex EMBED_TEXT_768 (snowflake-arctic-embed-m-v1.5), computed inside Snowflake.
# ---------------------------------------------------------
print("Embedding model:", EMBEDDING_LABEL)

# ---------------------------------------------------------
# Step 2: Embed the query and phrases from three intent groups (DOCX figure)
# ---------------------------------------------------------
query = "I want my money returned"
phrases = {
    "refund": "Payments / refunds", "chargeback": "Payments / refunds",
    "reimburse": "Payments / refunds", "money back": "Payments / refunds",
    "late delivery": "Delivery", "courier tracking": "Delivery",
    "reset my password": "Account access", "locked out": "Account access",
}
all_texts = [query] + list(phrases)
embeddings = embedding_model.encode(all_texts, convert_to_numpy=True)

print(f"Number of embeddings : {len(embeddings)}")
print(f"Embedding dimension  : {embeddings.shape[1]}   ->  E: text -> R^{embeddings.shape[1]} (model-specific)")
print(f"Sample vector (first 5 numbers of '{query}'): {embeddings[0][:5]}")

# ---------------------------------------------------------
# Step 3: Rank phrases by similarity to the query
# ---------------------------------------------------------
scores = cosine_similarity(embeddings[:1], embeddings[1:])[0]
query_words = set(query.lower().split())
ranking = pd.DataFrame({
    "Phrase": list(phrases),
    "Intent group": list(phrases.values()),
    "Shared keywords with query": [len(query_words & set(p.split())) for p in phrases],
    "Cosine similarity": scores.round(3),
}).sort_values("Cosine similarity", ascending=False)
display(ranking)

# Execution Topic: One-hot vs Dense Embeddings

With one-hot encoding every pair of words is equally different: refund·reimburse = 0, the same as refund·banana. Dense embeddings fix this by placing similar meanings close together.

This execution demonstrates a toy one-hot vocabulary next to real dense embeddings and connects it to the distributional hypothesis. The output will show identity (all dot products 0, all distances √2) for one-hot, versus meaningful similarity (refund ≈ reimburse, far from banana) for embeddings.

In [ ]:
# ---------------------------------------------------------
# Step 1: One-hot vectors for a toy vocabulary
# ---------------------------------------------------------
toy_vocab = ["refund", "reimburse", "banana"]
one_hot = np.eye(len(toy_vocab))
print("One-hot vectors:")
for word, vec in zip(toy_vocab, one_hot):
    print(f"  {word:<10} {vec}")

print("\nOne-hot dot products (meaning is NOT captured):")
display(pd.DataFrame(one_hot @ one_hot.T, index=toy_vocab, columns=toy_vocab))
print(f"Distance refund-reimburse = {np.linalg.norm(one_hot[0] - one_hot[1]):.3f}, "
      f"refund-banana = {np.linalg.norm(one_hot[0] - one_hot[2]):.3f}  -> equally distant")

# ---------------------------------------------------------
# Step 2: Dense embeddings for related vs unrelated sentences
# ---------------------------------------------------------
dense_sentences = ["I want a refund for my order", "Please reimburse the amount I paid", "I bought a banana"]
dense_vectors = embedding_model.encode(dense_sentences)
print("\nDense embedding cosine similarity (semantic geometry):")
display(pd.DataFrame(cosine_similarity(dense_vectors), index=["refund", "reimburse", "banana"],
                     columns=["refund", "reimburse", "banana"]).round(3))

# Execution Topic: Embedding Space Visualisation

Embeddings live in hundreds of dimensions; projecting them to 2-D with PCA gives an approximate picture of "meaning ⇒ position".

This execution demonstrates embedding the phrases from the DOCX figure and connects it to how related intents cluster together. The output will show payments, delivery and account-access phrases forming groups, with the query near the refund group (a 2-D projection loses some information, so treat it as an intuition).

In [ ]:
from sklearn.decomposition import PCA

# ---------------------------------------------------------
# Step 1: Phrases from the DOCX embedding-space figure
# ---------------------------------------------------------
groups = {
    "Payments / refunds": ["reimburse", "refund", "money back", "chargeback"],
    "Delivery": ["late delivery", "shipment", "courier", "tracking"],
    "Account access": ["password", "login", "OTP", "locked out"],
}
labels = [p for g in groups.values() for p in g]
vectors = embedding_model.encode(labels + [query])

# ---------------------------------------------------------
# Step 2: Project to 2-D and plot
# ---------------------------------------------------------
points = PCA(n_components=2, random_state=0).fit_transform(vectors)
plt.figure(figsize=(7, 5))
i = 0
for group_name, items in groups.items():
    xy = points[i:i + len(items)]
    plt.scatter(xy[:, 0], xy[:, 1], s=60, label=group_name)
    for (x, y), text in zip(xy, items):
        plt.annotate(text, (x, y), textcoords="offset points", xytext=(5, 5))
    i += len(items)
plt.scatter(*points[-1], marker="*", s=300, c="gold", edgecolors="black", label=f"Query: '{query}'")
plt.title("Embedding space (384-D projected to 2-D): meaning ⇒ position")
plt.legend(loc="best", fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Execution Topic: Cosine Similarity

Cosine similarity compares direction, not length: −1 opposite, 0 unrelated, +1 same direction.

$$\cos(\theta) = \frac{A \cdot B}{\lVert A \rVert \, \lVert B \rVert} = \frac{\sum a_i b_i}{\sqrt{\sum a_i^2}\,\sqrt{\sum b_i^2}}$$

This execution demonstrates the DOCX toy vectors A = "refund request", B = "money back please", C = "reset my password", computed manually with NumPy and verified with scikit-learn. The output will show cos(A,B) ≈ 0.985 (same intent) and cos(A,C) ≈ 0.14 (unrelated).

In [ ]:
# ---------------------------------------------------------
# Step 1: Toy 3-D embeddings from the DOCX
# ---------------------------------------------------------
A = np.array([0.8, 0.6, 0.1])     # "refund request"
B = np.array([0.7, 0.7, 0.2])     # "money back please"
C = np.array([-0.1, 0.2, 0.9])    # "reset my password"

# ---------------------------------------------------------
# Step 2: Manual calculation, step by step
# ---------------------------------------------------------
def cosine_manual(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

print(f"A·B = {np.dot(A, B):.2f} | ||A|| = {np.linalg.norm(A):.3f} | ||B|| = {np.linalg.norm(B):.3f}")
print(f"A·C = {np.dot(A, C):.2f} | ||C|| = {np.linalg.norm(C):.3f}")
print(f"\nManual  cos(A, B) = {cosine_manual(A, B):.3f}   <- near-identical intent")
print(f"Manual  cos(A, C) = {cosine_manual(A, C):.3f}   <- unrelated intent")

# ---------------------------------------------------------
# Step 3: Verify with scikit-learn
# ---------------------------------------------------------
sk = cosine_similarity([A], [B, C])[0]
print(f"\nsklearn cos(A, B) = {sk[0]:.3f}")
print(f"sklearn cos(A, C) = {sk[1]:.3f}")

# ---------------------------------------------------------
# Step 4: Length does not matter - scale B by 10, cosine is unchanged
# ---------------------------------------------------------
print(f"\ncos(A, 10*B) = {cosine_manual(A, 10 * B):.3f}  (a 'longer' text with the same meaning scores the same)")

# Execution Topic: Euclidean vs Cosine on Normalised Vectors

If vectors are normalised to length 1, cosine becomes a plain dot product and

$$\lVert A - B \rVert^2 = 2 - 2\cos(\theta)$$

so Euclidean distance and cosine produce the same ranking. This execution demonstrates the identity on the toy vectors and connects it to why vector databases normalise vectors and use the fast dot product. The output will show both sides of the equation matching and identical rankings.

In [ ]:
# ---------------------------------------------------------
# Step 1: Normalise the toy vectors to unit length
# ---------------------------------------------------------
def normalise(v):
    return v / np.linalg.norm(v)

A_n, B_n, C_n = normalise(A), normalise(B), normalise(C)
print("Lengths after normalising:", [round(float(np.linalg.norm(v)), 3) for v in (A_n, B_n, C_n)])

# ---------------------------------------------------------
# Step 2: Check ||A-B||^2 = 2 - 2cos(theta), and cos = dot product
# ---------------------------------------------------------
for name, other in [("B", B_n), ("C", C_n)]:
    cos_val = float(np.dot(A_n, other))
    dist_sq = float(np.sum((A_n - other) ** 2))
    print(f"A vs {name}: dot = {cos_val:.4f} | ||A-{name}||² = {dist_sq:.4f} | 2 - 2cos = {2 - 2 * cos_val:.4f}")

# ---------------------------------------------------------
# Step 3: Same ranking either way
# ---------------------------------------------------------
by_cosine = sorted(["B", "C"], key=lambda n: -np.dot(A_n, {"B": B_n, "C": C_n}[n]))
by_distance = sorted(["B", "C"], key=lambda n: np.linalg.norm(A_n - {"B": B_n, "C": C_n}[n]))
print("\nRanking by highest cosine    :", by_cosine)
print("Ranking by smallest distance :", by_distance)

# Execution Topic: Top-N Retrieval Ranking

Top-N keeps the N highest-scoring chunks after ranking by cosine similarity; those chunks go into the LLM's context.

$$\text{Top-N}(q) = \text{the } N \text{ chunks } d_i \text{ with the largest } \cos(E(q), E(d_i))$$

This execution demonstrates a small refund-policy knowledge base (sample data written for this notebook), embeds every chunk, scores it against a customer query and keeps `top_n = 3`. The output will show the ranked scores and which chunks enter the context.

In [ ]:
# ---------------------------------------------------------
# Step 1: A small sample knowledge base (written for this demo)
# ---------------------------------------------------------
knowledge_base = [
    "Refunds are credited to the original payment method within 5-7 business days after approval.",
    "Partial refunds: if only part of your refund has been credited, the remaining amount is credited within 7 business days of the first partial refund.",
    "A refund may be split into parts when an order was paid using two payment methods.",
    "Customers can raise a refund dispute if the credited amount does not match the amount paid.",
    "Electronics can be returned within 10 days of delivery.",
    "Items outside the return window can be covered by a warranty claim.",
    "Duplicate charges for the same order are reversed automatically within 48 hours.",
    "UPI payment page errors should be reported to the tech team with a screenshot.",
    "Delivery usually takes 3-5 business days and tracking is available in the app.",
    "Courier partners attempt delivery up to three times.",
    "To reset your password, use 'Forgot password' on the login page and enter the OTP.",
    "Accounts are locked for 30 minutes after 5 failed login attempts.",
]
kb_ids = [f"KB{i+1:02d}" for i in range(len(knowledge_base))]

# Embed every chunk ONCE (normalised, so cosine = dot product). Reused later.
kb_embeddings = embedding_model.encode(knowledge_base, normalize_embeddings=True)

# ---------------------------------------------------------
# Step 2: Score all chunks against the query, sort, keep Top-N
# ---------------------------------------------------------
def retrieve_top_n(query_text, top_n):
    q = embedding_model.encode([query_text], normalize_embeddings=True)
    scores = cosine_similarity(q, kb_embeddings)[0]
    order = np.argsort(-scores)
    return pd.DataFrame({"Chunk ID": [kb_ids[i] for i in order], "Score": scores[order].round(3),
                         "Chunk": [knowledge_base[i] for i in order]})

query_text = "When will I get the rest of my refund?"
top_n = 3
ranked = retrieve_top_n(query_text, top_n=len(knowledge_base))
print(f"Query: {query_text}\n\nTop-{top_n} chunks sent to the LLM context:")
display(ranked.head(top_n))

# ---------------------------------------------------------
# Step 3: Visualise the cut-off
# ---------------------------------------------------------
colors = ["tab:green" if i < top_n else "lightgrey" for i in range(len(ranked))]
plt.figure(figsize=(9, 3.5))
plt.bar(ranked["Chunk ID"], ranked["Score"], color=colors)
plt.axvline(top_n - 0.5, color="red", linestyle="--", label=f"Top-N cut-off (N={top_n})")
plt.ylabel("Cosine similarity"); plt.title("All chunks ranked; only the top N enter the context")
plt.legend(); plt.tight_layout(); plt.show()

# Execution Topic: Precision@N and Recall@N

Choosing N is a precision vs recall trade-off: small N is precise but may miss facts; large N finds everything but adds noise and cost.

$$\text{Precision@N} = \frac{\text{relevant in top }N}{N} \qquad \text{Recall@N} = \frac{\text{relevant in top }N}{\text{total relevant}}$$

This execution demonstrates the DOCX worked example (4 relevant chunks in the KB; N = 5 retrieves 3, N = 10 retrieves all 4) using illustrative relevance labels. The output will show P@5 = 0.60, R@5 = 0.75, P@10 = 0.40, R@10 = 1.0 and the full trade-off curve.

In [ ]:
# ---------------------------------------------------------
# Step 1: A ranked result list with relevance labels (1 = relevant)
# Illustrative labels built to match the DOCX worked example.
# ---------------------------------------------------------
ranked_relevance = [1, 0, 1, 1, 0, 0, 0, 1, 0, 0]   # rank 1 ... rank 10
total_relevant_in_kb = 4

def precision_at_n(relevance, n):
    return sum(relevance[:n]) / n

def recall_at_n(relevance, n, total_relevant):
    return sum(relevance[:n]) / total_relevant

# ---------------------------------------------------------
# Step 2: DOCX values
# ---------------------------------------------------------
for n in (5, 10):
    print(f"N = {n:>2}: relevant retrieved = {sum(ranked_relevance[:n])} | "
          f"Precision@{n} = {precision_at_n(ranked_relevance, n):.2f} | "
          f"Recall@{n} = {recall_at_n(ranked_relevance, n, total_relevant_in_kb):.2f}")
print("With N = 10 the LLM also reads", 10 - sum(ranked_relevance), "irrelevant chunks.")

# ---------------------------------------------------------
# Step 3: Trade-off curve for N = 1..10
# ---------------------------------------------------------
ns = range(1, 11)
plt.figure(figsize=(7, 3.5))
plt.plot(ns, [precision_at_n(ranked_relevance, n) for n in ns], "o-", label="Precision@N")
plt.plot(ns, [recall_at_n(ranked_relevance, n, total_relevant_in_kb) for n in ns], "o-", label="Recall@N")
plt.xlabel("N"); plt.ylabel("Score"); plt.title("Small N -> precision, large N -> recall")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Execution Topic: Retrieve Wide → Rerank Narrow

The production pattern is: fast cosine search retrieves many candidates (e.g. Top-50), a slower, more accurate reranker rescores them, and only the final Top-5 go to the LLM.

This execution demonstrates the three stages at small scale (retrieve 8 → rerank → keep 3) and connects it to getting recall from stage 1 and precision from stage 2. The reranking score is an **educational simulation** (cosine blended with query-term coverage), not a real reranker model. The output will show the candidate list before and after reranking.

In [ ]:
# ---------------------------------------------------------
# Small stop-word list, reused later for simple text checks
# ---------------------------------------------------------
STOPWORDS = {"a", "an", "the", "is", "are", "was", "were", "be", "been", "has", "have", "had", "i", "my", "me",
             "you", "your", "to", "of", "in", "on", "for", "and", "or", "it", "this", "that", "will", "can",
             "do", "does", "when", "what", "get", "got", "so", "also", "with", "by", "at", "only", "but", "if"}

def content_words(text):
    return [w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS]

def term_coverage(query_text, chunk_text):
    """Fraction of query content words found in the chunk (crude 5-letter prefix match)."""
    q_terms = content_words(query_text)
    c_words = content_words(chunk_text)
    hits = sum(any(w.startswith(t[:5]) for w in c_words) for t in q_terms)
    return hits / max(len(q_terms), 1)

# ---------------------------------------------------------
# Stage 1: Retrieve WIDE with fast cosine search (8 of 12 chunks here; 50 in production)
# ---------------------------------------------------------
rerank_query = "Only part of my refund was credited, when will I get the remaining amount?"
stage1 = retrieve_top_n(rerank_query, top_n=len(knowledge_base)).head(8).copy()
print("STAGE 1 - wide retrieval by cosine:")
display(stage1[["Chunk ID", "Score", "Chunk"]])

# ---------------------------------------------------------
# Stage 2: RERANK with a (simulated) more careful scorer
# ---------------------------------------------------------
stage1["Term coverage"] = [round(term_coverage(rerank_query, c), 2) for c in stage1["Chunk"]]
stage1["Rerank score (simulated)"] = (0.5 * stage1["Score"] + 0.5 * stage1["Term coverage"]).round(3)

# ---------------------------------------------------------
# Stage 3: Keep the final narrow Top-N
# ---------------------------------------------------------
final_top_n = 3
stage3 = stage1.sort_values("Rerank score (simulated)", ascending=False).head(final_top_n)
print(f"STAGE 2 + 3 - reranked, final Top-{final_top_n} for the LLM:")
display(stage3[["Chunk ID", "Score", "Term coverage", "Rerank score (simulated)", "Chunk"]])

## Part 4 — Generation Controls

# Execution Topic: Temperature and Softmax

The model outputs raw scores (logits) $z_i$; softmax turns them into probabilities, and temperature $T$ divides the logits first:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

This execution demonstrates the DOCX example "Your order has been ___" (delayed 4.0, late 3.2, stuck 2.5, lost 1.5, banana 0.5) at T = 0.2, 1.0 and 2.0. This is a mathematical sampling demonstration, no LLM is called. The output will show that low T sharpens, T = 1 keeps the native distribution, high T flattens, and every distribution sums to 1.

In [ ]:
# ---------------------------------------------------------
# Step 1: Toy logits from the DOCX
# ---------------------------------------------------------
next_tokens = ["delayed", "late", "stuck", "lost", "banana"]
logits = np.array([4.0, 3.2, 2.5, 1.5, 0.5])

def softmax_with_temperature(z, T):
    """p_i = exp(z_i / T) / sum_j exp(z_j / T).
    Subtracting the max first is a standard numerical-stability trick (result is identical)."""
    scaled = z / T
    exp_scores = np.exp(scaled - scaled.max())
    return exp_scores / exp_scores.sum()

# ---------------------------------------------------------
# Step 2: Probabilities at three temperatures
# ---------------------------------------------------------
temperatures = [0.2, 1.0, 2.0]
prob_table = pd.DataFrame({"Token": next_tokens, "Logit z": logits})
for T in temperatures:
    prob_table[f"z/T (T={T})"] = (logits / T).round(2)
    prob_table[f"P (T={T})"] = softmax_with_temperature(logits, T).round(3)
display(prob_table)

# ---------------------------------------------------------
# Step 3: Probabilities always sum to 1
# ---------------------------------------------------------
for T in temperatures:
    print(f"Temperature: {T}   Probability sum: {softmax_with_temperature(logits, T).sum():.4f}")

# Execution Topic: Temperature — Visual and Sampling

Seeing and sampling the distributions makes the effect concrete: at high temperature, nonsense like "banana" becomes plausible.

This execution demonstrates bar charts for the three temperatures and 1,000 simulated next-token draws at each, connecting to the DOCX guidance (low T for extraction/classification, higher T for creative text). The output will show the charts and how often "banana" is sampled at each temperature.

In [ ]:
# ---------------------------------------------------------
# Step 1: Bar charts, one per temperature
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, T in zip(axes, temperatures):
    p = softmax_with_temperature(logits, T)
    ax.bar(next_tokens, p)
    for i, v in enumerate(p):
        ax.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_title(f"T = {T}")
axes[0].set_ylabel("P(next token)")
fig.suptitle('"Your order has been ___" - same logits, different temperature')
plt.tight_layout(); plt.show()

# ---------------------------------------------------------
# Step 2: Simulate 1,000 next-token draws per temperature
# ---------------------------------------------------------
rng = np.random.default_rng(42)
for T in temperatures:
    draws = rng.choice(next_tokens, size=1000, p=softmax_with_temperature(logits, T))
    counts = Counter(draws)
    print(f"T={T}: 'delayed' {counts['delayed']:>4} times | 'banana' {counts['banana']:>3} times | "
          f"distinct tokens used: {len(counts)}")

# Execution Topic: Top-K Sampling

Top-K keeps only the K most probable tokens, removes the rest and renormalises:

$$p'_i = \frac{p_i}{\sum_{j \in V_K} p_j} \text{ if } i \in V_K, \quad \text{otherwise } p'_i = 0$$

This execution demonstrates the DOCX example with K = 3 (0.42, 0.24, 0.13 → sum 0.79) through a reusable `top_k_filter` function, connecting it to cutting off the long tail. The output will show 0.53, 0.30, 0.17 and that "banana"/"purple" can never be picked.

In [ ]:
# ---------------------------------------------------------
# Step 1: Toy next-token distribution (values from the DOCX figure)
# ---------------------------------------------------------
tokens_8 = ["delayed", "late", "stuck", "lost", "held", "moved", "banana", "purple"]
probs_8 = np.array([0.42, 0.24, 0.13, 0.08, 0.05, 0.04, 0.025, 0.015])

def top_k_filter(probabilities, k):
    """Keep the k highest probabilities, zero the rest, renormalise to sum to 1."""
    probabilities = np.asarray(probabilities, dtype=float)
    keep = np.argsort(-probabilities)[:k]
    filtered = np.zeros_like(probabilities)
    filtered[keep] = probabilities[keep]
    return filtered / filtered.sum()

# ---------------------------------------------------------
# Step 2: DOCX worked example, K = 3
# ---------------------------------------------------------
K = 3
kept_sum = np.sort(probs_8)[::-1][:K].sum()
print(f"Kept probabilities sum = {kept_sum:.2f}")
for t, p in zip(tokens_8[:K], probs_8[:K]):
    print(f"  {t:<8} {p:.2f} / {kept_sum:.2f} = {p / kept_sum:.3f}")
print("(The DOCX rounds these to 0.53, 0.30, 0.17 so that they add up to 1.00.)")

filtered_k = top_k_filter(probs_8, K)
display(pd.DataFrame({"Token": tokens_8, "Original p": probs_8, f"After Top-{K}": filtered_k.round(3)}))
print("Sum after filtering:", round(filtered_k.sum(), 4))

# ---------------------------------------------------------
# Step 3: Before vs after
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
axes[0].bar(tokens_8, probs_8, color=["tab:green"] * K + ["lightgrey"] * (len(tokens_8) - K))
axes[0].set_title(f"Original (green = kept by Top-{K})")
axes[1].bar(tokens_8, filtered_k, color="tab:green"); axes[1].set_title("Renormalised")
for ax in axes: ax.tick_params(axis="x", rotation=35)
plt.tight_layout(); plt.show()

# Execution Topic: Top-P (Nucleus) Sampling

Top-P keeps the smallest set of tokens whose cumulative probability reaches P, so the set size adapts to the model's confidence (unlike a fixed K).

This execution demonstrates P = 0.9 on the same distribution (0.42 + 0.24 + 0.13 + 0.08 + 0.05 = 0.92 ≥ 0.9 → 5 tokens) and then compares Top-K and Top-P on a confident and an uncertain distribution. The output will show which tokens each method keeps.

In [ ]:
# ---------------------------------------------------------
# Step 1: Top-P filter
# ---------------------------------------------------------
def top_p_filter(probabilities, p):
    """Keep the smallest set of highest-probability tokens whose cumulative sum >= p, renormalise."""
    probabilities = np.asarray(probabilities, dtype=float)
    order = np.argsort(-probabilities)
    cumulative = np.cumsum(probabilities[order])
    n_keep = int(np.searchsorted(cumulative, p - 1e-12) + 1)
    filtered = np.zeros_like(probabilities)
    filtered[order[:n_keep]] = probabilities[order[:n_keep]]
    return filtered / filtered.sum()

P = 0.9
display(pd.DataFrame({"Token": tokens_8, "p": probs_8, "Cumulative": np.cumsum(probs_8).round(3),
                      f"Kept by Top-P={P}": top_p_filter(probs_8, P) > 0}))

# ---------------------------------------------------------
# Step 2: Fixed K vs adaptive P on confident and uncertain distributions (toy values)
# ---------------------------------------------------------
confident = np.array([0.90, 0.04, 0.02, 0.015, 0.01, 0.008, 0.005, 0.002])
uncertain = np.array([0.16, 0.15, 0.14, 0.13, 0.12, 0.11, 0.10, 0.09])
for name, dist in [("Confident model", confident), ("Uncertain model", uncertain)]:
    kept_k = [t for t, v in zip(tokens_8, top_k_filter(dist, 3)) if v > 0]
    kept_p = [t for t, v in zip(tokens_8, top_p_filter(dist, P)) if v > 0]
    print(f"{name}:\n  Top-K (K=3)  keeps {len(kept_k)}: {kept_k}\n  Top-P (P=0.9) keeps {len(kept_p)}: {kept_p}")

# Execution Topic: Top-K vs Top-P vs Top-N

All three mean "keep the best few", but they act at different stages: Top-K and Top-P filter **next tokens** during generation, while Top-N ranks **documents** during retrieval.

This execution demonstrates a side-by-side comparison table and the generation pipeline order from the DOCX (logits → ÷ temperature → softmax → Top-K / Top-P filter → sample). The output will show the comparison and one full pass of the pipeline on the toy logits.

In [ ]:
# ---------------------------------------------------------
# Step 1: Comparison table
# ---------------------------------------------------------
comparison = pd.DataFrame([
    {"Setting": "Top-K", "Stage": "Token generation", "Ranks": "next tokens by probability",
     "Keeps": "fixed count K", "Example": "K = 3"},
    {"Setting": "Top-P", "Stage": "Token generation", "Ranks": "next tokens by probability",
     "Keeps": "smallest set with cumulative p >= P", "Example": "P = 0.9"},
    {"Setting": "Top-N", "Stage": "Document retrieval", "Ranks": "chunks by cosine similarity",
     "Keeps": "N best chunks for the context", "Example": "N = 3 (often called top_k in vector DBs)"},
])
display(comparison)

# ---------------------------------------------------------
# Step 2: Generation pipeline in the DOCX order
# ---------------------------------------------------------
def next_token_pipeline(z, T=0.7, k=3, p=0.9, seed=0):
    probs = softmax_with_temperature(z, T)          # logits -> / T -> softmax
    probs = top_k_filter(probs, k)                  # Top-K filter
    probs = top_p_filter(probs, p)                  # Top-P filter
    choice = np.random.default_rng(seed).choice(len(z), p=probs)   # sample
    return probs, choice

final_probs, chosen = next_token_pipeline(logits)
print("Pipeline: logits -> / temperature -> softmax -> Top-K / Top-P filter -> sample")
print("Final candidate probabilities:", {t: round(v, 3) for t, v in zip(next_tokens, final_probs) if v > 0})
print("Sampled next token:", next_tokens[chosen])

## Part 5 — Sentiment and Hallucination

# Execution Topic: Sentiment Value (−1 to +1) — Three Methods

A sentiment value is one number: −1 strongly negative, 0 neutral, +1 strongly positive. It can be sorted, thresholded and averaged, unlike a label.

$$\text{(1)}\; s = P(pos) - P(neg) \qquad \text{(2)}\; s = \frac{x}{\sqrt{x^2 + \alpha}},\; \alpha = 15 \qquad \text{(3)}\; s = \frac{r - 3}{2}$$

where $x$ is the sum of lexicon word scores and $r$ is a 1–5 star rating. This execution demonstrates all three DOCX worked examples for "Delivery was late and the box was damaged." The output will show −0.60 (classifier), −0.62 (lexicon) and the star-rating mapping.

In [ ]:
review = "Delivery was late and the box was damaged."

# ---------------------------------------------------------
# Method 1: Classifier probabilities (values from the DOCX)
# ---------------------------------------------------------
p_pos, p_neu, p_neg = 0.10, 0.20, 0.70
s_classifier = p_pos - p_neg
print(f"Method 1 (classifier): s = {p_pos:.2f} - {p_neg:.2f} = {s_classifier:.2f}")

# ---------------------------------------------------------
# Method 2: Lexicon (VADER-style). 'late' and 'damaged' scores are from the DOCX;
# the other entries are toy values used later in this notebook.
# ---------------------------------------------------------
toy_lexicon = {"late": -1.2, "damaged": -1.9, "great": 3.1, "bad": -2.5, "superb": 3.0,
               "terrible": -2.9, "ridiculous": -2.1, "love": 3.2}
ALPHA = 15

def lexicon_sentiment(text, lexicon=toy_lexicon, alpha=ALPHA):
    words = re.findall(r"[a-z]+", text.lower())
    x = sum(lexicon.get(w, 0.0) for w in words)
    return x, x / math.sqrt(x ** 2 + alpha)

x, s_lexicon = lexicon_sentiment(review)
print(f"Method 2 (lexicon)   : x = {x:.1f} -> s = {x:.1f} / sqrt({x**2:.2f} + {ALPHA}) = {s_lexicon:.2f}")

# ---------------------------------------------------------
# Method 3: Star rating
# ---------------------------------------------------------
print("Method 3 (stars)     : s = (r - 3) / 2")
display(pd.DataFrame({"Stars r": [1, 2, 3, 4, 5], "Sentiment s": [(r - 3) / 2 for r in range(1, 6)]}))

# Execution Topic: Sentiment Aggregation and Pitfalls

Businesses average sentiment across many texts; weighting by trust (e.g. helpful votes) can change the conclusion completely.

$$\bar{s}_{weighted} = \frac{\sum w_i s_i}{\sum w_i}$$

This execution demonstrates the DOCX reviews (+0.9, −0.8, +0.4, −0.2 with weights 1, 3, 1, 1) and connects it to business reporting: the simple mean (+0.075) hides that the most-trusted review is strongly negative (weighted −0.22). It also shows the DOCX pitfalls (sarcasm, negation, mixed reviews) with the toy lexicon.

In [ ]:
# ---------------------------------------------------------
# Step 1: Simple vs weighted mean (DOCX worked example 3)
# ---------------------------------------------------------
scores = np.array([0.9, -0.8, 0.4, -0.2])
weights = np.array([1, 3, 1, 1])     # helpful votes
simple_mean = scores.mean()
weighted_mean = np.sum(weights * scores) / np.sum(weights)
print(f"Simple mean   = {simple_mean:+.3f}   -> looks mildly positive")
print(f"Weighted mean = ({' + '.join(f'{w}×{s:+.1f}' for w, s in zip(weights, scores))}) / {weights.sum()} = {weighted_mean:+.2f}")
print("-> the most-trusted review is strongly negative; the weighted value reveals it.")

# ---------------------------------------------------------
# Step 2: Thresholding a signed value (e.g. escalate if s < -0.6)
# ---------------------------------------------------------
for s in [-0.72, -0.45, 0.30]:
    print(f"s = {s:+.2f} -> {'ESCALATE (high priority)' if s < -0.6 else 'normal queue'}")

# ---------------------------------------------------------
# Step 3: Pitfalls of word lists (DOCX examples)
# ---------------------------------------------------------
print("\nLexicon pitfalls:")
for text, truth in [("Great, another 2-hour wait!", "negative (sarcasm)"),
                    ("not bad", "mildly positive (negation)"),
                    ("Camera is superb, battery is terrible", "mixed (two strong aspects)")]:
    _, s = lexicon_sentiment(text)
    print(f"  {text:<40} lexicon s = {s:+.2f} | actually: {truth}")

# Execution Topic: Hallucination and a Groundedness Check

A hallucination is output that is incorrect or unsupported by the provided context or real-world facts, yet presented fluently and confidently.

This execution demonstrates answers with and without supporting context (simulated responses, including the DOCX contract-penalty example) and a simple deterministic groundedness check: numbers in a claim must appear in the evidence and most content words must be covered. **Educational simulation, not a production hallucination detector.** The output will show SUPPORTED vs UNSUPPORTED claims with reasons.

In [ ]:
# ---------------------------------------------------------
# Step 1: A simple, deterministic groundedness check (educational only)
# ---------------------------------------------------------
def extract_numbers(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text.replace(",", "")))

def check_claim(claim, evidence, min_overlap=0.5):
    missing_numbers = extract_numbers(claim) - extract_numbers(evidence)
    evidence_words = set(content_words(evidence))
    claim_words = content_words(claim)
    covered = [w for w in claim_words if any(e.startswith(w[:5]) or w.startswith(e[:5]) for e in evidence_words)]
    overlap = len(covered) / max(len(claim_words), 1)
    supported = not missing_numbers and overlap >= min_overlap
    reason = f"missing numbers {sorted(missing_numbers)}" if missing_numbers else f"word overlap {overlap:.0%}"
    return ("SUPPORTED" if supported else "UNSUPPORTED"), reason

# ---------------------------------------------------------
# Step 2: Return-policy example
# ---------------------------------------------------------
policy_context = "Refunds are allowed within 10 days."
print("Context :", policy_context)
print("Question: Can I return an item after 30 days?\n")
claims = [
    ("Answer WITH context (simulated)",    "No. Refunds are allowed only within 10 days."),
    ("Answer WITHOUT context (simulated)", "The company allows 30-day returns."),
]
for label, claim in claims:
    verdict, reason = check_claim(claim, policy_context)
    print(f"{label}\n  Claim: {claim}\n  -> {verdict} ({reason})\n")

# ---------------------------------------------------------
# Step 3: DOCX contract example - fluent, confident, but not in the contract
# ---------------------------------------------------------
contract_context = "Clause 7.3: a late-payment penalty of 2% applies after 30 days."
for label, claim in [("No context given (simulated guess)", "The late-payment penalty is 1.5% per month."),
                     ("Clause retrieved into context",      "The late-payment penalty is 2% after 30 days (Clause 7.3).")]:
    verdict, reason = check_claim(claim, contract_context)
    print(f"{label}: '{claim}' -> {verdict} ({reason})")

# Execution Topic: Hallucination Rate

Hallucination is measured on an evaluation set as the share of claims not supported by evidence:

$$\text{Hallucination rate} = \frac{\text{Unsupported claims}}{\text{Total claims}}$$

This execution demonstrates the metric on 10 annotated claims (2 unsupported → 20%) and then with other counts, connecting it to tracking hallucination over time. The output will show how the rate changes as the numbers change.

In [ ]:
# ---------------------------------------------------------
# Step 1: A tiny annotated eval set (True = supported by evidence)
# ---------------------------------------------------------
claim_is_supported = [True, True, False, True, True, True, False, True, True, True]
total_claims = len(claim_is_supported)
unsupported_claims = claim_is_supported.count(False)
rate = unsupported_claims / total_claims
print(f"Total claims = {total_claims} | Unsupported = {unsupported_claims} | Hallucination rate = {rate:.0%}")

# ---------------------------------------------------------
# Step 2: Change the numbers and watch the metric
# ---------------------------------------------------------
scenarios = [(10, 2), (10, 0), (50, 5), (200, 30)]
display(pd.DataFrame({
    "Total claims": [t for t, _ in scenarios],
    "Unsupported claims": [u for _, u in scenarios],
    "Hallucination rate": [f"{u / t:.1%}" for t, u in scenarios],
}))

## Part 6 — Reasoning and Prompting

# Execution Topic: Step-by-Step Reasoning (Insurance Claim)

Reasoning means working through a problem in steps (extract, apply rule, calculate, check) instead of jumping to an answer. Each written step becomes context for the next.

This execution demonstrates the DOCX claim: 80% cover after a ₹5,000 deductible, cap ₹1,00,000, claim ₹32,000. The output will show the auditable steps giving ₹21,600, the one-jump shortcut giving the wrong ₹20,600, and a large claim where the cap check matters.

In [ ]:
# ---------------------------------------------------------
# Helper: Indian number formatting (₹1,00,000)
# ---------------------------------------------------------
def inr(amount):
    s = str(int(round(amount)))
    if len(s) <= 3:
        return "₹" + s
    head, tail = s[:-3], s[-3:]
    groups = []
    while len(head) > 2:
        groups.insert(0, head[-2:]); head = head[:-2]
    if head:
        groups.insert(0, head)
    return "₹" + ",".join(groups + [tail])

# ---------------------------------------------------------
# Step-by-step reasoning with a visible, auditable log
# ---------------------------------------------------------
def insurance_payout_stepwise(claim, deductible=5_000, coverage=0.80, cap=100_000, verbose=True):
    eligible = max(claim - deductible, 0)
    payout = coverage * eligible
    final = min(payout, cap)
    if verbose:
        print(f"Step 1 (extract): claim = {inr(claim)}, deductible = {inr(deductible)}, coverage = {coverage:.0%}, cap = {inr(cap)}")
        print(f"Step 2 (apply rule): eligible = {inr(claim)} - {inr(deductible)} = {inr(eligible)}")
        print(f"Step 3 (calculate): payout = {coverage:.2f} × {inr(eligible)} = {inr(payout)}")
        print(f"Step 4 (check): {inr(payout)} {'<=' if payout <= cap else '>'} cap {inr(cap)} -> final = {inr(final)}")
    return final

correct = insurance_payout_stepwise(32_000)

# ---------------------------------------------------------
# The one-jump shortcut: applies 80% BEFORE the deductible (wrong order)
# ---------------------------------------------------------
shortcut = 0.80 * 32_000 - 5_000
print(f"\nShortcut: 0.8 × {inr(32_000)} - {inr(5_000)} = {inr(shortcut)}  <- wrong by {inr(correct - shortcut)}")

# ---------------------------------------------------------
# A large claim: Step 4 (cap check) changes the answer
# ---------------------------------------------------------
print("\nLarge claim:")
insurance_payout_stepwise(150_000)

# Execution Topic: Reasoning Reliability (pⁿ)

Even with a high per-step accuracy, long chains fail often, because every step must be right:

$$P(\text{all } n \text{ steps correct}) = p^n$$

This execution demonstrates p = 0.95 for 1–10 steps (0.95⁵ ≈ 0.77, 0.95¹⁰ ≈ 0.60) and connects it to the DOCX safeguards: tool verification and majority voting (self-consistency). The output will show the decay curve, a tool check catching the shortcut answer, and a simplified majority-vote calculation.

In [ ]:
# ---------------------------------------------------------
# Step 1: Chain reliability p^n
# ---------------------------------------------------------
p_step = 0.95
steps = np.arange(1, 11)
chain_reliability = p_step ** steps
print(f"0.95^5  = {p_step ** 5:.2f}")
print(f"0.95^10 = {p_step ** 10:.2f}")

plt.figure(figsize=(7, 3.2))
plt.plot(steps, chain_reliability, "o-")
plt.xlabel("Number of steps n"); plt.ylabel("P(all steps correct)")
plt.title("Chain reliability with p = 0.95 per step"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# ---------------------------------------------------------
# Step 2: Verification with a tool (Python acts as the calculator)
# ---------------------------------------------------------
model_answer = 20_600                                                   # e.g. a one-jump answer
tool_answer = insurance_payout_stepwise(32_000, verbose=False)          # deterministic calculation
print(f"\nVerification: model said {inr(model_answer)}, tool computed {inr(tool_answer)} -> "
      f"{'OK' if model_answer == tool_answer else 'MISMATCH - flag for review'}")

# ---------------------------------------------------------
# Step 3: Majority voting over 5 reasoning paths (simplified: independent paths,
# each fully correct with p = 0.60, and wrong answers do not agree with each other)
# ---------------------------------------------------------
p_path, n_paths = p_step ** 10, 5
p_majority = sum(math.comb(n_paths, k) * p_path ** k * (1 - p_path) ** (n_paths - k) for k in range(3, n_paths + 1))
print(f"Single path correct: {p_path:.2f} | Majority of {n_paths} paths correct (simplified): {p_majority:.2f}")

# Execution Topic: Zero-shot Prompting

Zero-shot means performing a task using only instructions, with no examples: $\hat{y} = \arg\max P(y \mid \text{instruction}, x)$.

This execution demonstrates the DOCX prompt "Classify the sentiment … Battery died in 2 days." and separates **prompt design** (real) from **inference** (simulated by simple keyword rules that output label scores). The output will show the prompt, the simulated label probabilities and the argmax label: Negative.

> **Note:** This is an educational simulation of an LLM response. No external LLM API is being called.

In [ ]:
# ---------------------------------------------------------
# Step 1: Prompt design (this part is real - it is what you would send)
# ---------------------------------------------------------
LABELS = ["Positive", "Negative", "Neutral"]

def zero_shot_prompt(review_text):
    return (f"Classify the sentiment of this review as {', '.join(LABELS[:-1])} or {LABELS[-1]}.\n"
            f'Review: "{review_text}"')

# ---------------------------------------------------------
# Step 2: SIMULATED inference - keyword rules produce P(label | instruction, x)
# ---------------------------------------------------------
def mock_zero_shot_probs(review_text):
    text = review_text.lower()
    neg = sum(w in text for w in ["died", "broken", "late", "worst", "damaged"])
    pos = sum(w in text for w in ["love", "great", "excellent", "fast"])
    scores = np.array([pos, neg, 0.5]) * 2.0          # toy logits
    return softmax_with_temperature(scores, 1.0)

for review_text in ["Battery died in 2 days.", "Love it, super fast!", "Order shipped today."]:
    probs = mock_zero_shot_probs(review_text)
    print(zero_shot_prompt(review_text))
    print("  Simulated P(label):", {l: round(float(p), 2) for l, p in zip(LABELS, probs)},
          "-> output:", LABELS[int(np.argmax(probs))], "\n")

# Execution Topic: Zero-shot Limitations (Company Taxonomy)

Zero-shot works for generic tasks but breaks down for company-specific labels, strict output formats, domain jargon and borderline business rules.

This execution demonstrates what happens when a reasonable generic answer meets a company taxonomy (P1-Billing-Refund-Partial, P2-Billing-Duplicate, P3-Tech-Payment-UI) and a strict JSON format. Generic outputs are simulated. The output will show that sensible labels are still rejected by the routing system.

In [ ]:
# ---------------------------------------------------------
# Step 1: The company's exact taxonomy and required output format
# ---------------------------------------------------------
COMPANY_TAXONOMY = {"P1-Billing-Refund-Partial", "P2-Billing-Duplicate", "P3-Tech-Payment-UI"}
REQUIRED_FORMAT = 'JSON like {"label": "<taxonomy label>"}'

# ---------------------------------------------------------
# Step 2: Plausible zero-shot outputs (SIMULATED) vs what routing needs
# ---------------------------------------------------------
cases = [
    ("Refund only half credited",       "Refund issue",                   "P1-Billing-Refund-Partial"),
    ("Charged twice for order 88",      "The label is: Payment problem",  "P2-Billing-Duplicate"),
    ("App crashes on UPI page",         '{"label": "App bug"}',           "P3-Tech-Payment-UI"),
]

def router_accepts(model_output):
    """Routing only accepts valid JSON whose label is in the taxonomy."""
    try:
        return json.loads(model_output).get("label") in COMPANY_TAXONOMY
    except (json.JSONDecodeError, AttributeError):
        return False

rows = [{"Ticket": t, "Zero-shot output (simulated)": out, "Required label": req,
         "Valid JSON?": out.strip().startswith("{"), "Router accepts?": router_accepts(out)} for t, out, req in cases]
display(pd.DataFrame(rows))
print("Required format:", REQUIRED_FORMAT)
print("Every answer is 'reasonable', yet none can be routed -> the model needs to SEE the taxonomy and format.")

# Execution Topic: Few-shot Prompting

Few-shot adds a few worked examples (input → output) to the prompt; the model continues the pattern without any weight change (in-context learning):

$$\hat{y} = \arg\max P(y \mid \text{instruction}, (x_1,y_1), \dots, (x_k,y_k), x)$$

This execution demonstrates the DOCX billing-taxonomy prompt and the new ticket "Got ₹200 back, paid ₹500". Prompt construction is real; the answer is **simulated** by a hand-written rule standing in for the pattern an LLM would infer from the examples. The output will show the full prompt and the label BILLING_PARTIAL_REFUND in the exact format.

In [ ]:
# ---------------------------------------------------------
# Step 1: Few-shot examples from the DOCX
# ---------------------------------------------------------
few_shot_examples = [
    ("Charged twice for order 88", "BILLING_DUPLICATE"),
    ("Refund only half credited", "BILLING_PARTIAL_REFUND"),
    ("App crashes on UPI page", "TECH_PAYMENT_UI"),
]
FEW_SHOT_LABELS = {label for _, label in few_shot_examples}

def build_few_shot_prompt(ticket, examples):
    lines = ["Label the ticket."]
    lines += [f'"{x}" → {y}' for x, y in examples]
    lines.append(f'Ticket: "{ticket}" →')
    return "\n".join(lines)

# ---------------------------------------------------------
# Step 2: SIMULATED model - a rule that mimics the pattern shown by the examples
# ---------------------------------------------------------
def mock_few_shot_label(ticket):
    text = ticket.lower()
    amounts = [int(a.replace(",", "")) for a in re.findall(r"₹\s?([\d,]+)", ticket)]
    if "twice" in text or "double" in text:
        return "BILLING_DUPLICATE"
    if "crash" in text or "upi" in text:
        return "TECH_PAYMENT_UI"
    if ("half" in text or "part" in text) or (len(amounts) >= 2 and min(amounts) < max(amounts)):
        return "BILLING_PARTIAL_REFUND"
    return "UNKNOWN"

new_ticket = "Got ₹200 back, paid ₹500"
prompt = build_few_shot_prompt(new_ticket, few_shot_examples)
label = mock_few_shot_label(new_ticket)

print(prompt)
print(f"\n[Simulated model output] {label}")
print("In the taxonomy?", label in FEW_SHOT_LABELS, "| Zero-shot would likely say 'Refund issue' -> not routable")

# Execution Topic: Few-shot Cost

Examples are paid for on every request:

$$\text{Extra prompt cost} \approx k \times \text{(tokens per example)} \times \text{(number of requests)}$$

This execution demonstrates the DOCX numbers (5 examples × 60 tokens = 300 extra tokens/call; 1,00,000 calls/day → 30M tokens/day), priced with the notebook's **illustrative** $3/M input price. The output will show the extra tokens and cost for different k, plus the real token size of our example lines.

In [ ]:
# ---------------------------------------------------------
# Step 1: DOCX assumptions
# ---------------------------------------------------------
tokens_per_example = 60
num_examples_k = 5
requests_per_day = 100_000
extra_per_request = num_examples_k * tokens_per_example
extra_per_day = extra_per_request * requests_per_day

print(f"Tokens per example   : {tokens_per_example}")
print(f"Number of examples k : {num_examples_k}")
print(f"Extra tokens/request : {extra_per_request}")
print(f"Requests per day     : {requests_per_day:,}")
print(f"Extra tokens per day : {extra_per_day:,}  (= {extra_per_day / 1e6:.0f}M)")
print(f"Illustrative extra cost at ${price_input_per_million}/M input: ${extra_per_day / 1e6 * price_input_per_million:,.2f}/day")

# ---------------------------------------------------------
# Step 2: How cost grows with k (typical range 2-8)
# ---------------------------------------------------------
display(pd.DataFrame({"k": range(0, 9),
                      "Extra tokens/day (M)": [k * tokens_per_example * requests_per_day / 1e6 for k in range(9)],
                      "Illustrative $/day": [k * tokens_per_example * requests_per_day / 1e6 * price_input_per_million for k in range(9)]}))

# ---------------------------------------------------------
# Step 3: Real size of our short example lines (cl100k_base)
# ---------------------------------------------------------
for x, y in few_shot_examples:
    line = f'"{x}" → {y}'
    print(f"{len(enc_cl100k.encode(line)):>3} tokens: {line}")

# Execution Topic: Dynamic Few-shot (Retrieve Similar Examples)

For large label sets, instead of fixed examples, retrieve the most similar labelled examples for each request using embeddings + cosine similarity.

This execution demonstrates embedding a small labelled ticket set, finding the top-3 matches for "Only part of my refund was credited", and turning them into few-shot examples. It is a retrieval demonstration; no LLM is called. The output will show each example's similarity and label, and the resulting prompt.

In [ ]:
# ---------------------------------------------------------
# Step 1: Small labelled ticket history (sample data)
# ---------------------------------------------------------
labelled_tickets = pd.DataFrame([
    ("Charged twice for order 88", "BILLING_DUPLICATE"),
    ("My card was debited two times for one purchase", "BILLING_DUPLICATE"),
    ("Refund only half credited", "BILLING_PARTIAL_REFUND"),
    ("I received only part of my refund amount", "BILLING_PARTIAL_REFUND"),
    ("Got ₹200 back, paid ₹500", "BILLING_PARTIAL_REFUND"),
    ("App crashes on UPI page", "TECH_PAYMENT_UI"),
    ("Payment screen freezes when I choose UPI", "TECH_PAYMENT_UI"),
    ("My parcel is 5 days late", "DELIVERY_DELAY"),
    ("Courier has not delivered yet", "DELIVERY_DELAY"),
    ("Cannot log in, OTP not arriving", "ACCOUNT_LOGIN"),
], columns=["Example", "Label"])
example_vectors = embedding_model.encode(labelled_tickets["Example"].tolist(), normalize_embeddings=True)

# ---------------------------------------------------------
# Step 2: Retrieve the most similar labelled examples for the new ticket
# ---------------------------------------------------------
def retrieve_examples(ticket, k=3):
    q = embedding_model.encode([ticket], normalize_embeddings=True)
    result = labelled_tickets.copy()
    result["Similarity"] = cosine_similarity(q, example_vectors)[0].round(3)
    return result.sort_values("Similarity", ascending=False).head(k)[["Example", "Similarity", "Label"]]

incoming_ticket = "Only part of my refund was credited"
dynamic_examples = retrieve_examples(incoming_ticket, k=3)
print(f"New ticket: {incoming_ticket}")
display(dynamic_examples)

# ---------------------------------------------------------
# Step 3: The retrieved examples become the few-shot block of the prompt
# ---------------------------------------------------------
print(build_few_shot_prompt(incoming_ticket, list(zip(dynamic_examples["Example"], dynamic_examples["Label"]))))

## Part 7 — Putting Everything Together

# Execution Topic: Pipeline Steps 1–4 — Understand and Retrieve

A customer writes: **"Paid ₹500 but only got ₹200 back, this is ridiculous!"** The following three executions trace this one request through every concept.

This execution demonstrates tokenization/token count, sentiment value (classifier probabilities are **simulated**), embedding + cosine similarity against the knowledge base, and Top-N = 4 retrieval (real embeddings). The output will show the token count, sentiment and priority, and the four policy chunks selected. Real cosine scores will differ from the illustrative values in the theory document.

In [ ]:
customer_message = "Paid ₹500 but only got ₹200 back, this is ridiculous!"

# ---------------------------------------------------------
# Step 1: Tokenization / token count (real tokenizer)
# ---------------------------------------------------------
message_token_count = len(enc_cl100k.encode(customer_message))
print(f"[1] Tokens: {message_token_count} (cl100k_base) - counted for cost and window budget")

# ---------------------------------------------------------
# Step 2: Sentiment value s = P(pos) - P(neg)  (SIMULATED classifier probabilities)
# ---------------------------------------------------------
sim_p_pos, sim_p_neu, sim_p_neg = 0.05, 0.18, 0.77
sentiment_value = round(sim_p_pos - sim_p_neg, 2)
priority = "HIGH" if sentiment_value < -0.6 else "NORMAL"
print(f"[2] Sentiment s = {sim_p_pos} - {sim_p_neg} = {sentiment_value} -> priority {priority} (threshold -0.6)")

# ---------------------------------------------------------
# Steps 3 + 4: Embedding + cosine similarity -> Top-N policy chunks (real embeddings)
# ---------------------------------------------------------
PIPELINE_TOP_N = 4
retrieved = retrieve_top_n(customer_message, top_n=len(knowledge_base)).head(PIPELINE_TOP_N)
print(f"[3] Message embedded ({embedding_model.get_sentence_embedding_dimension()}-D) and compared with {len(knowledge_base)} KB chunks")
print(f"[4] Top-{PIPELINE_TOP_N} policy chunks:")
display(retrieved)

# Execution Topic: Pipeline Steps 5–9 — Context, Few-shot, Reasoning, Decoding, Grounding

This execution demonstrates assembling the context (system rules + order record + retrieved chunks + few-shot examples + question) and checking it fits the window with output reserved, labelling with few-shot, reasoning the pending amount with a tool, greedy decoding at temperature → 0 with Top-K (toy label logits, **simulated**), and a groundedness check on the draft reply. The output will show the token budget, ₹300 pending, the label, and a hallucinated claim being blocked.

In [ ]:
# ---------------------------------------------------------
# Step 5: Context construction + context-window check
# ---------------------------------------------------------
order_record = "Order A1029: amount paid ₹500; amount refunded so far ₹200; first partial refund credited 3 days ago."
system_rules = ("You are a support assistant. Answer only from the policy and order record. "
                "Reply as JSON with a label from the taxonomy. If the policy does not cover the case, escalate.")
context_text = "\n".join([
    "SYSTEM: " + system_rules,
    "ORDER RECORD: " + order_record,
    "POLICY CHUNKS:\n" + "\n".join(f"[{cid}] {c}" for cid, c in zip(retrieved["Chunk ID"], retrieved["Chunk"])),
    "EXAMPLES:\n" + build_few_shot_prompt(customer_message, few_shot_examples),
])
context_tokens, output_reserve_tokens = len(enc_cl100k.encode(context_text)), 1_000
print(f"[5] Context tokens: {context_tokens} + output reserve {output_reserve_tokens} "
      f"<= {N_MAX:,}? {fits_in_window(context_tokens, output_reserve_tokens)}")

# ---------------------------------------------------------
# Step 6: Few-shot label (SIMULATED model, rule stands in for pattern-following)
# ---------------------------------------------------------
predicted_label = mock_few_shot_label(customer_message)
print(f"[6] Few-shot label: {predicted_label}")

# ---------------------------------------------------------
# Step 7: Reasoning - extract amounts, calculate with a tool, check the policy timeline
# ---------------------------------------------------------
paid, refunded = 500, 200
pending_refund = paid - refunded
timeline_chunks = [c for c in retrieved["Chunk"] if "remaining amount" in c]
policy_covers_case = bool(timeline_chunks)
timeline_days = int(re.search(r"within (\d+) business days", timeline_chunks[0]).group(1)) if policy_covers_case else None
tool_result = f"Calculated by tool: pending remaining refund = ₹500 - ₹200 = ₹{pending_refund}"
print(f"[7] Reasoning: pending = {inr(paid)} - {inr(refunded)} = {inr(pending_refund)}; "
      f"policy timeline found: {policy_covers_case} ({timeline_days} business days)")

# ---------------------------------------------------------
# Step 8: Temperature -> 0 (greedy) + Top-K on toy label logits (SIMULATED)
# ---------------------------------------------------------
label_candidates = ["BILLING_PARTIAL_REFUND", "BILLING_DUPLICATE", "Refund issue", "TECH_PAYMENT_UI"]
label_logits = np.array([5.0, 2.0, 1.5, 0.5])
p_low_t = top_k_filter(softmax_with_temperature(label_logits, 0.2), k=2)
print(f"[8] T=0.2 + Top-K=2 -> {({l: round(float(p), 4) for l, p in zip(label_candidates, p_low_t)})}")
print(f"    Greedy pick (T -> 0): {label_candidates[int(np.argmax(p_low_t))]} - predictable, parseable, no long-tail tokens")

# ---------------------------------------------------------
# Step 9: Groundedness check of the draft reply (plus one injected hallucination)
# ---------------------------------------------------------
evidence = " ".join([order_record, tool_result] + list(retrieved["Chunk"]))
draft_claims = [
    "You paid ₹500 and ₹200 has been refunded so far.",
    "The remaining ₹300 is still pending.",
]
if policy_covers_case:   # only cite a timeline if the retrieved policy actually contains one
    draft_claims.append(f"The remaining amount is credited within {timeline_days} business days of the first partial refund.")
draft_claims.append("You will also receive a ₹100 goodwill voucher.")   # injected hallucination
claim_results = [(c, *check_claim(c, evidence)) for c in draft_claims]
display(pd.DataFrame(claim_results, columns=["Claim", "Verdict", "Reason"]))
final_claims = [c for c, v, _ in claim_results if v == "SUPPORTED"]
print(f"[9] {len(draft_claims) - len(final_claims)} unsupported claim removed before replying.")

# Execution Topic: Final Structured Response

The last step assembles a parseable, grounded reply and decides whether a human is needed (escalate only if the policy doesn't cover the case or a claim can't be grounded).

This execution demonstrates the final JSON a downstream system would consume and a readable pipeline summary. The output will show every stage's result in one place.

> **Note:** This is a mock pipeline built from real tokenizers/embeddings and simulated LLM steps. It is not a real LLM.

In [ ]:
# ---------------------------------------------------------
# Step 1: Decide escalation and build the structured (JSON) response
# ---------------------------------------------------------
all_grounded = len(final_claims) > 0 and all(check_claim(c, evidence)[0] == "SUPPORTED" for c in final_claims)
escalate_to_human = not policy_covers_case or not all_grounded
cited_chunk = retrieved.loc[retrieved["Chunk"].isin(timeline_chunks), "Chunk ID"].tolist()

structured_response = {
    "label": predicted_label,
    "priority": priority,
    "pending_refund_inr": pending_refund,
    "reply": " ".join(final_claims),
    "citations": cited_chunk,
    "grounded": all_grounded,
    "escalate": escalate_to_human,
}

# ---------------------------------------------------------
# Step 2: Print the pipeline summary
# ---------------------------------------------------------
line = "=" * 60
print(line); print("LLM SUPPORT ASSISTANT — PIPELINE DEMO (simulated LLM steps)"); print(line)
print(f"\nCustomer message:\n  {customer_message}")
print(f"\nToken count:\n  {message_token_count} tokens (cl100k_base)")
print(f"\nSentiment:\n  s = {sentiment_value} -> priority {priority}")
print(f"\nTop-{PIPELINE_TOP_N} retrieved policy chunks:")
for cid, score, chunk in zip(retrieved["Chunk ID"], retrieved["Score"], retrieved["Chunk"]):
    print(f"  [{cid}] {score:.3f}  {chunk[:75]}{'...' if len(chunk) > 75 else ''}")
print("\nFew-shot examples:")
for x, y in few_shot_examples:
    print(f'  "{x}" → {y}')
print(f"\nPending refund:\n  {inr(pending_refund)}")
print(f"\nFinal classification:\n  {predicted_label}")
print(f"\nGroundedness:\n  {'SUPPORTED' if all_grounded else 'UNSUPPORTED'} (hallucinated claim removed)")
print(f"\nEscalation:\n  {'YES' if escalate_to_human else 'NO'}")
print(f"\nStructured response (JSON):\n{json.dumps(structured_response, indent=2, ensure_ascii=False)}")
print(line)

### Final discussion
- Which single step in this pipeline would hurt most if it failed silently, and how would you monitor it?
- If the policy chunk about partial refunds were **not** retrieved, what should the assistant do? (Try it: set `PIPELINE_TOP_N = 1` and re-run Part 7.)

## Recap: Concept → Formula → Execution

| Concept | Key formula / rule | Where |
|---|---|---|
| Token | 1 token ≈ 4 chars ≈ 0.75 words; Cost = (Tᵢₙ·Pᵢₙ + Tₒᵤₜ·Pₒᵤₜ)/1M; Latency ≈ TTFT + Tₒᵤₜ·TPOT | Part 1 |
| Tokenization | BPE: merge argmax count(a, b) | Part 1 |
| Context | P(y₁…yₘ \| context) = ∏ P(yₜ \| context, y₍<ₜ₎) | Part 2 |
| Context window | Tᵢₙ + Tₒᵤₜ ≤ Nₘₐₓ; cost ∝ n²; chunks = ⌈(L − o)/(s − o)⌉ | Part 2 |
| Embedding / Cosine | E: text → ℝᵈ; cos = A·B/(‖A‖‖B‖); ‖A−B‖² = 2 − 2cos | Part 3 |
| Top-N | Precision@N, Recall@N; retrieve wide → rerank narrow | Part 3 |
| Temperature / Top-K / Top-P | pᵢ = e^(zᵢ/T)/Σe^(zⱼ/T); p′ᵢ = pᵢ/Σ_{Vₖ} pⱼ | Part 4 |
| Sentiment | P(pos) − P(neg); x/√(x²+α); (r − 3)/2; weighted mean | Part 5 |
| Hallucination | unsupported claims / total claims | Part 5 |
| Reasoning | pⁿ | Part 6 |
| Zero-shot / Few-shot | argmax P(y \| instruction, [examples], x); k × tokens × requests | Part 6 |